# IST Scenario 5 — Who survives a stroke? One survival model

_Four predictors, two descriptive looks, one Cox regression. Level: first model, no statistics background needed. About 30 minutes._

**What you will do.** Choose an outcome and four possible predictors, look at them with two quick
descriptive analyses, then run a single model, a *Cox Proportional Hazards Regression*, that weighs
all four predictors at once. On the way you will meet a classic trap: in IST, women died more
often than men after their stroke. Is being a woman a risk, or is something else hiding behind it?

**Words you will meet.** *Follow-up time*: the days from entering the trial to death or, for a
patient still alive, to the last news of them (about six months later). *Censored*: a patient
still alive at the last news; a survival model uses the time they were known to be alive instead
of throwing it away. *Hazard*: the risk of dying at a given moment among the patients still alive.
*Hazard ratio*: how many times higher the hazard is in one group than in a reference group; 2
means twice as high, 0.5 half as high, 1 no difference; for age, per year. *Reference group*: here
men, fully alert patients and patients without atrial fibrillation. A *crude* comparison takes the
groups as they are, an *adjusted* one as if they were alike in the other predictors. *Atrial
fibrillation*: an irregular heartbeat in which clots can form in the heart and travel to the
brain.

**Before you start.** IST randomised 19,435 patients between 1991 and 1996. Nobody received
thrombolysis or thrombectomy, one third were randomised before a CT scan, and "dependent" was
measured by questionnaire, not by the modified Rankin Scale. The *conclusions* of the trial still
hold and are current practice; the *absolute percentages* describe 1990s stroke care and must not
be quoted to patients.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model).
No patient row ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

## 1. Connect to the MIP and open the IST pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("IST")      # add version="1.0" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())        # the pathology label differs: pick it from this list
    raise
ist = dm.datasets.list()[0]             # the IST pathology holds a single dataset
display(dm)
display(ist)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    out = {}
    for item in raw:
        if isinstance(item, dict):
            out[str(item.get("code"))] = str(item.get("label"))
        else:
            out[str(item)] = str(item)
    return out


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the IST dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ist], variables=variables), filters=filters,
                        handle_missing=handler)


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == "all datasets"]


def numeric_summary(result, variables):
    """Mean, SD, median and range of numeric variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "median": d.get("q2"),
                         "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def describe_by(variables, group, base_filter=None):
    """Describe the variables separately in each level of `group` (one federated call per level).
    Returns (numeric table, categorical table) with a column per group level."""
    num, cat = [], []
    for c, label in labels_of(group).items():
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(variables, f).describe()
        n = numeric_summary(res, [v for v in variables if v.is_numerical()])
        if len(n):
            num.append(n.assign(group=label))
        for v in variables:
            if v.is_categorical():
                k = category_counts(res, v)
                if len(k):
                    cat.append(k.assign(group=label))
    num = pd.concat(num, ignore_index=True) if num else pd.DataFrame()
    cat = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
    if len(num):
        num = num.pivot(index="variable", columns="group", values=["n", "mean", "sd"])
    if len(cat):
        cat = cat.pivot(index=["variable", "level"], columns="group", values="percent")
    return num, cat


def crosstab(exposure, outcome, base_filter=None, event="1"):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c, label in lab_x.items():
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: label, "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.2f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(5, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first level of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

In [ ]:
# ---- Toolbox, part 2: helpers for the Cox model and its forest plot. Run this cell once. ----
def term_label(term, variables):
    """Model term -> readable label: 'RXHEP[2]' -> 'Heparin allocated = Medium dose', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level).split(':')[0]}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def hazard_ratio_table(result, variables):
    """Cox regression result -> one row per term with hazard ratio, 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    s = raw.get("summary", {})
    terms = raw.get("indep_vars") or s.get("indep_vars") or []
    return pd.DataFrame({"term": [term_label(t, variables) for t in terms],
                         "hazard_ratio": s.get("hazard_ratios"), "ci_low": s.get("hr_lower_ci"),
                         "ci_high": s.get("hr_upper_ci"), "p": s.get("pvalues")})


def forest(ax, table, ratio, title):
    """Forest plot into ax: one ratio (column odds_ratio or hazard_ratio) per term with its 95% CI."""
    t = table.iloc[::-1].reset_index(drop=True)
    ax.errorbar(t[ratio], t.index, xerr=[t[ratio] - t["ci_low"], t["ci_high"] - t[ratio]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ticks = [x for x in (0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100) if t["ci_low"].min() <= x <= t["ci_high"].max()]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{x:g}" for x in ticks])
    ax.minorticks_off()
    ax.set_yticks(t.index)
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(ratio.replace("_", " ") + " (log scale)")
    ax.set_title(title, fontsize=10)


print("Toolbox part 2 ready.")

## Step 1. Choose the variables

The outcome of a survival model has two parts: the follow-up time (`Time to death or censoring`)
and whether the patient died (`Died`). The four predictors are things any doctor notes at
admission: age, sex, level of consciousness and atrial fibrillation.

| Code | Label in the MIP | Tree location (under IST) | Role | Values |
|---|---|---|---|---|
| `TD` | Time to death or censoring | Death and survival / Survival analysis | outcome: follow-up time | integer, 0-1400 days |
| `DIED` | Died | Death and survival / Survival analysis | outcome: the event, death | 0 No; 1 Yes |
| `AGE` | Age | Demographics and event / Demographics | predictor | integer, 16-99 years |
| `SEX` | Sex | Demographics and event / Demographics | predictor | 1 Male; 2 Female |
| `RCONSC` | Conscious level | Clinical assessment at admission / Consciousness and vitals | predictor | 1 Fully alert; 2 Drowsy; 3 Unconscious |
| `RATRIAL` | Atrial fibrillation | Prestroke characteristics / Cerebrovascular risk factors | predictor; empty in the pilot phase (984 patients) | 0 No; 1 Yes |

In [ ]:
td      = dm.variables["Time to death or censoring"]     # outcome: follow-up time
died    = dm.variables["Died"]                           # outcome: the event
age     = dm.variables["Age"]                            # the four predictors
sex     = dm.variables["Sex"]
rconsc  = dm.variables["Conscious level"]
ratrial = dm.variables["Atrial fibrillation"]

display(mip.to_frame([td, died, age, sex, rconsc, ratrial]))

## Step 2. Describe them

*Expected:* 19,435 patients; age 71.7 years on average (median 73); 46.5% women; 76.8% fully
alert, 21.9% drowsy, 1.3% unconscious; atrial fibrillation in 17.2% of the 18,451 patients in
whom it was recorded (not recorded for the 984 patients of the pilot phase); 4,370 deaths
(22.5%); time to death or censoring, median 186 days. A typical IST patient was in their
seventies and alert, and was followed for about six months; about one in five died during that
time.

In [ ]:
whole = analysis([td, died, age, sex, rconsc, ratrial]).describe()
display(numeric_summary(whole, [age, td]).style.format({"mean": "{:.1f}", "sd": "{:.1f}"}).hide(axis="index"))
for v in (sex, rconsc, ratrial, died):
    display(category_counts(whole, v).style.format({"percent": "{:.1f}"}).hide(axis="index"))

## Step 3. A first look: do women fare worse?

One `describe()` per sex for age and death, then the age distribution of each sex as a histogram.

*Expected:* men 10,407, of whom 2,093 died (20.1%), mean age 69.6 years; women 9,028, of whom
2,277 died (25.2%), mean age 74.2 years. Women died more often, by about 5 percentage points. But
women were also older, by almost five years on average, and older patients die more often. From
these numbers alone we cannot tell whether sex or age is behind the difference. A model that looks
at all predictors together can.

In [ ]:
num, cat = describe_by([age, died], group=sex)          # one describe per sex
display(num.style.format("{:.1f}"))
display(cat.style.format("{:.1f}"))

colours = {"Male": "#4C72B0", "Female": "#C44E52"}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [2, 1]})
for level, label in labels_of(sex).items():
    h = analysis([age], F(sex) == level, drop_missing=True).histogram(variable=age, bins=20).raw["histogram"][0]
    counts = np.array([c or 0 for c in h["counts"]], dtype=float)
    axes[0].bar(h["bins"][:-1], 100 * counts / counts.sum(), width=np.diff(h["bins"]), align="edge",
                alpha=0.5, color=colours.get(label), label=label)
axes[0].set_xlabel("age (years)")
axes[0].set_ylabel("% of the patients of each sex")
axes[0].set_title("Women were older")
axes[0].legend()

died_pct = cat.loc[(died.label, "Yes")]                 # % died, one value per sex
axes[1].bar(died_pct.index, died_pct.values, color=[colours.get(g) for g in died_pct.index])
for i, value in enumerate(died_pct.values):
    axes[1].text(i, value + 0.5, f"{value:.1f}%", ha="center")
axes[1].set_ylim(0, 1.2 * died_pct.max())                # room for the labels
axes[1].set_ylabel("% died during follow-up")
axes[1].set_title("Women died more often")
plt.tight_layout()
plt.show()

## Step 4. One model: Cox regression

In the MIP interface: filter `Time to death or censoring > 0` (55 patients died on the day they
entered the trial; the algorithm needs a follow-up time above zero); outcome `Time to death or
censoring`; predictors `Died`, `Age`, `Sex`, `Conscious level`, `Atrial fibrillation`; then *Cox
Proportional Hazards Regression* with event variable `Died` and event of interest Yes (code 1).
Each category is compared with its first code: men, fully alert, no atrial fibrillation. The forest plot shows one dot per predictor with its 95% confidence interval; the
dashed line at 1 means no difference.

*Expected (18,395 patients, 4,105 deaths), hazard ratios:* women 0.92 (95% CI 0.86 to 0.98);
drowsy 3.27 (3.07 to 3.48) and unconscious 8.77 (7.53 to 10.22) versus fully alert; atrial
fibrillation 1.49 (1.39 to 1.60); age 1.051 per year, 1.64 per 10 years. About 1,040 patients
drop out: the 984 pilot-phase patients without atrial fibrillation data and the 57 without a
follow-up time above zero.

- **Consciousness is the strongest predictor.** A drowsy patient died more than three times as
  fast as an alert one, an unconscious patient almost nine times as fast: a reduced level of
  consciousness is the sign of a large stroke.
- **Age:** every ten years of age multiply the hazard by about 1.6.
- **Atrial fibrillation** multiplies it by 1.5: these strokes are often large, and the patients
  often have heart disease.
- **Women:** compared at the same age, level of consciousness and heart rhythm, women died
  slightly *less* fast than men (0.92). Their higher crude death rate came from being older, more
  often drowsy or unconscious (27% against 20%) and more often in atrial fibrillation (21% against
  14%). This is **confounding**: age made sex look like a risk.

In [ ]:
covariates = [age, sex, rconsc, ratrial]
model = analysis([td, died] + covariates, F(td) > 0, drop_missing=True).cox_regression_classical(
    time=td, event_var=died, covariates=covariates, positive_class="1")
print({k: model.raw["summary"].get(k) for k in ("n_obs", "n_events")})

hrs = hazard_ratio_table(model, covariates)
hrs["term"] = hrs["term"].str.replace("(per unit)", "(per year)", regex=False)
display(hrs.style.format({"hazard_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": fmt_p}).hide(axis="index"))
per_year = hrs[hrs["term"].str.startswith("Age")].iloc[0]
print(f"Age, per 10 years: {per_year['hazard_ratio'] ** 10:.2f} ({per_year['ci_low'] ** 10:.2f} to {per_year['ci_high'] ** 10:.2f})")

fig, ax = plt.subplots(figsize=(8, 3.6))
forest(ax, hrs, "hazard_ratio", "Who died sooner? Compared with men, fully alert, no atrial fibrillation")
plt.tight_layout()
plt.show()

## Step 5 (if time allows). Watch the confounding disappear

The same model three times, always on the same patients as Step 4: with sex alone, with sex and
age, then with all four predictors.

*Expected:* women versus men 1.30 (1.23 to 1.39) with sex alone, 1.00 (0.94 to 1.06) once age is
added, 0.92 (0.86 to 0.98) with all four. The women's excess deaths disappear as soon as the
model compares women and men of the same age. In the MIP interface, keep the experiment and
change only the predictors: atrial fibrillation stays in the experiment, so preprocessing keeps
the same patients and the numbers are the same.

In [ ]:
same_patients = (F(td) > 0) & F(ratrial).is_not_null()           # the patients of the Step 4 model
models = {"sex only": [sex], "sex + age": [sex, age], "all four (Step 4)": covariates}
rows = []
for name, covs in models.items():
    res = analysis([td, died] + covs, same_patients, drop_missing=True).cox_regression_classical(
        time=td, event_var=died, covariates=covs, positive_class="1")
    w = hazard_ratio_table(res, covs).set_index("term").loc["Sex = Female"]
    rows.append({"model": name, **w.to_dict()})
women = pd.DataFrame(rows)
display(women.style.format({"hazard_ratio": "{:.2f}", "ci_low": "{:.2f}", "ci_high": "{:.2f}", "p": fmt_p}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.errorbar(range(len(women)), women["hazard_ratio"], fmt="o-", capsize=6,
            yerr=[women["hazard_ratio"] - women["ci_low"], women["ci_high"] - women["hazard_ratio"]])
ax.axhline(1, linestyle="--", linewidth=1, color="grey")
ax.set_xticks(range(len(women)))
ax.set_xticklabels(women["model"])
ax.set_ylabel("hazard ratio, women vs men")
ax.set_title("The women's excess disappears once age is in the model")
plt.tight_layout()
plt.show()

## The result in two sentences

> Among 18,395 IST patients followed for about six months, the hazard of death rose with age
> (1.64 per 10 years), was more than three times higher in drowsy and almost nine times higher in
> unconscious patients than in alert ones, and 1.5 times higher with atrial fibrillation. Women
> died more often than men (25% against 20%), but only because they were older and had more severe
> strokes: compared like for like, their hazard was slightly lower (0.92).

**Be careful.** This is an observational question asked of trial data: the model shows who was at
risk, not what caused the deaths or what would prevent them. A hazard ratio is an average over the
whole follow-up; for unconscious patients most of the excess deaths happened in the first days.
These are 1990s patients, before stroke units, thrombolysis and thrombectomy were routine.

**Questions for discussion**

1. A newspaper headline says "Women more likely to die after a stroke". Using your results, write
   a better headline.
2. Consciousness is the strongest predictor. Is it a cause of death, or a sign of something else?
3. The model says who is at risk. Does it say what to do? What kind of study would show that
   treating atrial fibrillation saves lives?
4. The 984 pilot-phase patients drop out of the model. Could that change the results? How would you
   check?